# 연습 2: 다른 환경의 사진에서도 잘 맞히나? (교차 데이터셋 테스트)

> ⚠️ **제출용이 아닙니다.** 공개 데이터셋으로 하는 연습입니다. 여기서 만든 가중치·사진은 제출하지 않고, 교수님 점수 확인 서버에도 보내지 않습니다.

**질문:** 내 사진으로 학습하고 내 사진으로 검증하면 점수가 높게 나와도, 교수님이 다른 곳에서 찍은 사진에서는 떨어질 수 있다(오버피팅/환경 차이). 얼마나 떨어지고, 무엇이 그 차이를 줄이나?

**방법:** 서로 다른 사람이 다른 환경에서 모은 두 데이터셋을 씁니다.

| 이름 | 데이터 | 특징 | 클래스 대응 |
|---|---|---|---|
| **VN** | [VN Trash Classification](https://www.kaggle.com/datasets/mrgetshjtdone/vn-trash-classification) (Kaggle, MIT) | 베트남 대학 연구팀. 실내 촬영 + 인터넷 제품 사진, 한국 우유팩도 있음 | `Alu`→can, `Milk_box`→carton, `PET`→pet (`Carton` 폴더는 골판지라 안 씀) |
| **TACO** | [TACO](http://tacodataset.org) (Hugging Face 사본) | 길바닥·풀밭에 버려진 쓰레기 | Drink can→can, Drink carton→carton, plastic bottle→pet |

- **같은 환경 점수**: VN으로 학습 → VN 테스트 사진으로 평가 (hw02에서 검증 정확도를 보는 것과 같음)
- **다른 환경 점수**: VN으로 학습 → TACO 사진으로 평가 (교수님 사진으로 채점받는 상황과 같음)

모델은 hw02와 똑같습니다: 동결한 ResNet-18 + `fc`(512→3)만 학습, 평가 전처리는 `tf`.

**Colab 설정:** 런타임 → 런타임 유형 변경 → T4 GPU. Kaggle 키(`kaggle.json`)가 필요합니다: Kaggle → Settings → API → *Create New Token*.

In [ ]:
import os, json, random
import numpy as np
import torch, torch.nn as nn
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, ConcatDataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CLASSES = ['can', 'carton', 'pet']
print(device)

## 1. VN 데이터셋 내려받기 (Kaggle, 약 2GB)

처음 실행하면 `kaggle.json` 파일을 올리라는 버튼이 나옵니다. 키는 이 Colab 런타임에만 저장됩니다.

In [ ]:
VN_DIR = './vn/VN_trash_classification'
if not os.path.isdir(VN_DIR):
    kj = os.path.expanduser('~/.kaggle/kaggle.json')
    if not os.path.exists(kj) and 'KAGGLE_KEY' not in os.environ:
        from google.colab import files
        print('kaggle.json 파일을 골라 주세요')
        up = files.upload()
        os.makedirs(os.path.dirname(kj), exist_ok=True)
        open(kj, 'wb').write(up['kaggle.json'])
        os.chmod(kj, 0o600)
    !kaggle datasets download -d mrgetshjtdone/vn-trash-classification -p ./vn --unzip -q
print(sorted(os.listdir(VN_DIR + '/Train')))

## 2. TACO에서 can / carton / pet 잘라내기

연습 1 노트북의 0절과 같은 코드입니다. 이미 `./practice_photos`가 있으면 건너뜁니다.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from huggingface_hub import hf_hub_download

PHOTO_DIR = './practice_photos'
if not os.path.isdir(PHOTO_DIR):
    REPO = 'pictograph/taco'
    TACO_TO_HW02 = {'Drink can': 'can', 'Drink carton': 'carton',
                    'Clear plastic bottle': 'pet', 'Other plastic bottle': 'pet'}
    MIN_BOX, PAD = 40, 0.15
    meta = hf_hub_download(REPO, 'train/metadata.jsonl', repo_type='dataset')
    jobs = {}
    for line in open(meta):
        r = json.loads(line)
        for box, name in zip(r['objects']['bbox'], r['objects']['category_names']):
            if name in TACO_TO_HW02 and min(box[2], box[3]) >= MIN_BOX:
                jobs.setdefault(r['file_name'], []).append((TACO_TO_HW02[name], box))

    def crop_one(item):
        file_name, boxes = item
        img = ImageOps.exif_transpose(Image.open(
            hf_hub_download(REPO, 'train/' + file_name, repo_type='dataset'))).convert('RGB')
        out = []
        for c, (x, y, w, h) in boxes:
            crop = img.crop((max(0, x - w * PAD), max(0, y - h * PAD),
                             min(img.width, x + w * (1 + PAD)), min(img.height, y + h * (1 + PAD))))
            s = 384 / min(crop.size)
            if s < 1:
                crop = crop.resize((round(crop.width * s), round(crop.height * s)))
            out.append((c, crop))
        return file_name, out

    for c in CLASSES:
        os.makedirs(os.path.join(PHOTO_DIR, c), exist_ok=True)
    with ThreadPoolExecutor(8) as ex:
        for file_name, crops in ex.map(crop_one, sorted(jobs.items())):
            stem = file_name.replace('/', '_').rsplit('.', 1)[0]
            for k, (c, crop) in enumerate(crops):
                crop.save(os.path.join(PHOTO_DIR, c, f'{stem}_{k}.jpg'), quality=90)
print({c: len(os.listdir(os.path.join(PHOTO_DIR, c))) for c in CLASSES})

## 3. 평가용·학습용 폴더 만들기

모든 사진을 hw02 2절처럼 짧은 변 256으로 줄여 `./cross/<이름>/<클래스>/`에 저장합니다.

| 폴더 | 내용 | 용도 |
|---|---|---|
| `vn_train` | VN Train에서 클래스마다 `N_VN`장 | 학습 |
| `vn_test` | VN Test 전부 (클래스마다 약 100장) | 같은 환경 평가 |
| `taco_a`, `taco_b` | TACO를 **원본 사진 단위로** 반씩 나눔 | `taco_a`는 섞어서 학습할 때, `taco_b`는 다른 환경 평가 |

TACO는 한 사진에서 물건 여러 개를 잘라냈기 때문에, 같은 사진에서 나온 조각이 학습과 평가에 동시에 들어가지 않도록 **원본 사진 단위로** 나눕니다. (hw02에서 같은 물건을 여러 장 찍었을 때 생기는 문제와 같은 이야기입니다.)

In [ ]:
VN_MAP = {'Alu': 'can', 'Milk_box': 'carton', 'PET': 'pet'}
N_VN = 300          # VN 학습 사진 클래스당 장수 (내가 찍을 장수와 비슷한 규모)
ROOT = './cross'
random.seed(42)

def save_small(src, dst):
    try:
        img = ImageOps.exif_transpose(Image.open(src)).convert('RGB')
    except Exception:
        return False                       # 깨진 파일은 건너뜀
    s = 256 / min(img.size)
    img = img.resize((max(1, round(img.width * s)), max(1, round(img.height * s))))
    img.save(dst, quality=90)
    return True

def build(name, items):                    # items: [(원본 경로, 클래스)]
    for c in CLASSES:
        os.makedirs(os.path.join(ROOT, name, c), exist_ok=True)
    n = 0
    for k, (src, c) in enumerate(items):
        n += save_small(src, os.path.join(ROOT, name, c, f'{k:05d}.jpg'))
    print(name, {c: len(os.listdir(os.path.join(ROOT, name, c))) for c in CLASSES})

if not os.path.isdir(ROOT):
    train_items, test_items = [], []
    for vn, c in VN_MAP.items():
        names = sorted(os.listdir(f'{VN_DIR}/Train/{vn}'))
        random.shuffle(names)
        train_items += [(f'{VN_DIR}/Train/{vn}/{n}', c) for n in names[:N_VN]]
        test_items += [(f'{VN_DIR}/Test/{vn}/{n}', c) for n in sorted(os.listdir(f'{VN_DIR}/Test/{vn}'))]
    build('vn_train', train_items)
    build('vn_test', test_items)

    taco = [(os.path.join(PHOTO_DIR, c, n), c) for c in CLASSES for n in sorted(os.listdir(os.path.join(PHOTO_DIR, c)))]
    stems = sorted({os.path.basename(p).rsplit('_', 1)[0] for p, _ in taco})   # 원본 사진 이름
    random.shuffle(stems)
    half = set(stems[:len(stems) // 2])
    build('taco_a', [(p, c) for p, c in taco if os.path.basename(p).rsplit('_', 1)[0] in half])
    build('taco_b', [(p, c) for p, c in taco if os.path.basename(p).rsplit('_', 1)[0] not in half])

In [ ]:
# 각 폴더에서 클래스마다 다섯 장씩 보기
fig, axes = plt.subplots(3, 10, figsize=(16, 5))
for r, c in enumerate(CLASSES):
    for k, name in enumerate(['vn_train', 'taco_b']):
        files = sorted(os.listdir(f'{ROOT}/{name}/{c}'))[:5]
        for j, f in enumerate(files):
            ax = axes[r, k * 5 + j]
            ax.imshow(Image.open(f'{ROOT}/{name}/{c}/{f}')); ax.set_title(f'{name}\n{c}', fontsize=8); ax.axis('off')
plt.tight_layout(); plt.show()

## 4. 전처리, 증강, 실험 함수

`tf`는 hw02와 같습니다(채점기 전처리). 증강은 연습 1의 8절과 같은 세 가지를 비교합니다.

In [ ]:
mean, std = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
T = transforms
end = [T.ToTensor(), T.Normalize(mean, std)]
tf = T.Compose([T.Resize(256), T.CenterCrop(224)] + end)

AUG = {
    'A 증강 없음': tf,
    'C 중간 증강': T.Compose([T.RandomResizedCrop(224, scale=(0.4, 1.0)), T.RandomHorizontalFlip(),
                           T.RandomRotation(20), T.ColorJitter(0.3, 0.3, 0.3, 0.05)] + end),
    'D 강한 증강': T.Compose([T.RandomResizedCrop(224, scale=(0.2, 1.0)), T.RandomHorizontalFlip(),
                           T.RandomRotation(30), T.ColorJitter(0.4, 0.4, 0.4, 0.1),
                           T.RandomGrayscale(0.1)] + end),
}
EVAL = {name: DataLoader(datasets.ImageFolder(f'{ROOT}/{name}', transform=tf), batch_size=64)
        for name in ['vn_test', 'taco_b']}
weights = models.ResNet18_Weights.IMAGENET1K_V1

def scores(m, loader):
    conf = np.zeros((3, 3), dtype=int)
    m.eval()
    with torch.no_grad():
        for xb, yb in loader:
            pred = m(xb.to(device)).argmax(1).cpu()
            for t, p in zip(yb, pred):
                conf[t, p] += 1
    per = conf.diagonal() / conf.sum(1)
    return conf.trace() / conf.sum(), per

results = []
def run(name, train_sets, aug, epochs=15, lr=0.001):
    torch.manual_seed(42)
    ds = ConcatDataset([datasets.ImageFolder(f'{ROOT}/{s}', transform=AUG[aug]) for s in train_sets])
    loader = DataLoader(ds, batch_size=32, shuffle=True, num_workers=2)
    m = models.resnet18(weights=weights)
    for p in m.parameters():
        p.requires_grad = False
    m.fc = nn.Linear(512, 3)
    m = m.to(device)
    opt = torch.optim.Adam(m.fc.parameters(), lr=lr)
    crit = nn.CrossEntropyLoss()
    for e in range(epochs):
        m.train()
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad(); crit(m(xb), yb).backward(); opt.step()
    row = {'실험': name, '학습': '+'.join(train_sets), '증강': aug, '학습 장수': len(ds)}
    line = f'{name:28s}'
    for ev, ld in EVAL.items():
        acc, per = scores(m, ld)
        row[ev] = acc; row[ev + ' 클래스별'] = per
        line += f' | {ev}: 전체 {acc:.3f} (can {per[0]:.2f} carton {per[1]:.2f} pet {per[2]:.2f})'
    results.append(row)
    print(line)

## 5. 실험

1~3: **VN으로만 학습**하고 증강만 바꿈 → 같은 환경(`vn_test`)과 다른 환경(`taco_b`) 점수 차이 보기
4: VN에 **다른 환경 사진(`taco_a`)을 섞어** 학습 → `taco_b`가 좋아지나? (= 사진을 다양한 환경에서 찍으면 좋아지나?)
5: **TACO 조금(`taco_a`)만**으로 학습 → 적고 한 가지 환경의 사진이면 어떻게 되나?

In [ ]:
run('1 VN, 증강 없음',        ['vn_train'], 'A 증강 없음')
run('2 VN, 중간 증강',        ['vn_train'], 'C 중간 증강')
run('3 VN, 강한 증강',        ['vn_train'], 'D 강한 증강')
run('4 VN+TACO-A, 중간 증강', ['vn_train', 'taco_a'], 'C 중간 증강')
run('5 TACO-A만, 중간 증강',  ['taco_a'], 'C 중간 증강')

In [ ]:
# 같은 환경 vs 다른 환경 점수 비교
names = [r['실험'] for r in results]
x = np.arange(len(results))
plt.figure(figsize=(10, 4))
plt.bar(x - 0.2, [r['vn_test'] for r in results], 0.4, label='VN 테스트 (VN과 같은 환경)')
plt.bar(x + 0.2, [r['taco_b'] for r in results], 0.4, label='TACO-B (다른 환경)')
plt.xticks(x, names, rotation=15, fontsize=8); plt.ylim(0, 1); plt.ylabel('accuracy'); plt.legend()
plt.show()
for r in results:
    print(f"{r['실험']:28s} 학습 {r['학습 장수']:4d}장 | VN 테스트 {r['vn_test']:.3f} | TACO-B {r['taco_b']:.3f} | 차이 {r['vn_test'] - r['taco_b']:+.3f}")

## 6. hw02에 주는 교훈 (결과를 보고 확인할 것)

- **같은 환경 점수와 다른 환경 점수의 차이**가 클수록, hw02 검증 정확도가 실제 점수(교수님 사진)를 부풀려 보여 준다는 뜻입니다.
- 실험 4가 실험 2보다 TACO-B에서 좋다면 → **여러 환경에서 찍은 사진을 섞는 것**이 다른 환경 일반화에 도움이 된다는 증거입니다. hw02에서는 장소·배경·조명을 다양하게 찍는 것이 이에 해당합니다.
- 증강(1~3)이 다른 환경 점수를 얼마나 바꾸는지 보세요. 데이터를 다양하게 모으는 것보다 효과가 작다면, 튜닝보다 촬영에 시간을 쓰는 게 낫습니다.
- 실험 5는 사진이 적고 한 환경뿐일 때의 모습입니다.